In [ ]:
# DO NOT MODIFY

import torch
import triton
import triton.language as tl
import time

# Data Helper
def make_input(n):
    buffer = torch.full((n + 64,), 1_000_000.0, dtype=torch.float32, device='cuda')

    active_region = torch.arange(1, n + 1, dtype=torch.float32, device='cuda')
    buffer[:n] = active_region

    return buffer[:n]

# Ground truth
def sum_rows_torch(x):
    return torch.sum(x)

In [ ]:
# Kernel and launcher
@triton.jit
def sum_rows_kernel(
    x_ptr,
    output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr
):

    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    x = tl.load(x_ptr + offsets)

    block_sum = tl.sum(x, axis=0)
    tl.store(output_ptr + pid, block_sum)


def sum_rows(x):
    n = x.numel()
    BLOCK_SIZE = 128

    grid = n // BLOCK_SIZE

    print(f"grid: {grid}")

    if grid == 0:
        return torch.tensor(0.0, device=x.device)

    partial_sums = torch.zeros(grid, dtype=torch.float32, device=x.device)

    sum_rows_kernel[(grid,)](x, partial_sums, n, BLOCK_SIZE=BLOCK_SIZE)

    return torch.sum(partial_sums)

In [ ]:
# Kernel and launcher (Updated definition to account for arbitrary input size)
@triton.jit
def sum_rows_kernel(
    x_ptr,
    output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr
):

    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements # ignores indices beyond end of input
    x = tl.load(x_ptr + offsets, mask=mask, other=0)

    block_sum = tl.sum(x, axis=0)
    tl.store(output_ptr + pid, block_sum)


def sum_rows(x):
    n = x.numel()
    BLOCK_SIZE = 128

    grid = (n + BLOCK_SIZE - 1) // BLOCK_SIZE # take ceil instead of floor

    print(f"grid: {grid}")

    if grid == 0:
        return torch.tensor(0.0, device=x.device)

    partial_sums = torch.zeros(grid, dtype=torch.float32, device=x.device)

    sum_rows_kernel[(grid,)](x, partial_sums, n, BLOCK_SIZE=BLOCK_SIZE)

    return torch.sum(partial_sums)

In [ ]:
# Running on powers of two
def test_aligned_sizes():
    sizes = [256, 512, 1024, 4096]

    for n in sizes:
        x = make_input(n)
        expected = sum_rows_torch(x).item()
        actual = sum_rows(x).item()

        assert expected == actual, f"Failed at n={n}: expected {expected}, got {actual}"
        print(f"PASS: n={n}")

test_aligned_sizes()

grid: 2
PASS: n=256
grid: 4
PASS: n=512
grid: 8
PASS: n=1024
grid: 32
PASS: n=4096


In [ ]:
# Run for n = 1000 here
def test_n_1000():
    n = 1000
    x = make_input(n)
    expected = sum_rows_torch(x).item()
    actual = sum_rows(x).item()

    assert expected == actual, f"Failed at n={n}: expected {expected}, got {actual}"
    print(f"PASS: n={n}")
test_n_1000()

grid: 7


AssertionError: Failed at n=1000: expected 500500.0, got 401856.0

In [ ]:
# Timing baseline
def chewie_sum(x):
    """
    A chunked Python loop to act as baseline implementation.
    """
    chunk_size = 128
    total = 0.0
    cpu_x = x.cpu()
    for i in range(0, cpu_x.numel(), chunk_size):
        chunk = cpu_x[i:i+chunk_size]
        total += sum(chunk.tolist())
    return total

In [ ]:
# Timing - Harness A
def time_it_A(n=4096, runs=10):
    x = make_input(n)

    start = time.time()
    for _ in range(runs):
        _ = chewie_sum(x)
    baseline_time = (time.time() - start) / runs

    start = time.time()
    for _ in range(runs):
        _ = sum_rows(x)
    candidate_time = (time.time() - start) / runs

    print("Harness A")
    print(f"chewie_sum: {baseline_time*1000:.4f} ms")
    print(f"sum_rows:   {candidate_time*1000:.4f} ms")
    print(f"Reported Speedup: {baseline_time / candidate_time:.2f}x")

time_it_A()

grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
Harness A
chewie_sum: 0.4313 ms
sum_rows:   0.1075 ms
Reported Speedup: 4.01x


In [ ]:
# Timing - Harness B
def time_it_B(n=4096, runs=100):
    x = make_input(n)

    # 1. Warm-up
    for _ in range(5):
        chewie_sum(x)
        sum_rows(x)

    # 2. Time baseline properly
    baseline_times = []
    for _ in range(runs):
        start = time.time()
        chewie_sum(x)
        baseline_times.append(time.time() - start)

    # 3. Time candidate using CUDA events
    candidate_times = []
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    for _ in range(runs):
        start_event.record()
        sum_rows(x)
        end_event.record()
        torch.cuda.synchronize()
        candidate_times.append(start_event.elapsed_time(end_event))

    # 4. Median to ignore OS jitter outliers
    baseline_median = sorted(baseline_times)[runs // 2] * 1000
    candidate_median = sorted(candidate_times)[runs // 2]

    print(f"Harness B")
    print(f"chewie_sum (baseline): {baseline_median:.4f} ms")
    print(f"sum_rows (candidate):  {candidate_median:.4f} ms")
    print(f"Reported Speedup: {baseline_median / candidate_median:.2f}x")

time_it_B()

grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
grid: 32
Harness B
chewie_sum (baseline): 0.2048 ms
sum_rows (ca